# Data Extraction Flow — External Public Data (RF CNPJ → CVM → PGFN)

| step | module | obs |
|---|---|---|
| Receita Federal CNPJ | `pipelines.cnpj` | latest release; full release does not fit the quota — use `cnpj_ufs` |
| CVM DFP/ITR | `pipelines.cvm` | current year |
| PGFN devedores | `pipelines.pgfn` | latest quarter; `pgfn_ufs` optional |

Sequential, *continue on error*. CGU, MTE, PNCP and CAGED have their own notebooks.

In [ ]:
import pandas as pd
from pipelines.flow import run_flow
from lakehouse import core as C

pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 60)

## Inputs

In [ ]:
target_db = "estudo"           # pasta de destino no bucket lake (lake/estudo/)
cnpj_ufs = ["ES"]          # [] = Brasil inteiro (~7 GB zipados: não cabe na cota de 5 GiB)
cnpj_tables = None         # None = todas; ex.: ["Estabelecimentos", "Empresas", "Socios", "Cnaes"]
pgfn_ufs = []              # [] = todas as UFs (~1.4 GB zipados)
dry_run = True              # True = só plano + verificação leve das fontes

## Execute

In [ ]:
spark = None if dry_run else C.get_spark("flow_empresas", cores_max=4)
resumo = run_flow(spark, target_db=target_db, cnpj_ufs=cnpj_ufs, cnpj_tables=cnpj_tables, pgfn_ufs=pgfn_ufs, dry_run=dry_run)
pd.DataFrame(resumo)

## Bucket usage

The bucket has a hard quota (5 GiB by default); every pipeline estimates its write and refuses one that would not fit.

In [ ]:
C.quota_check(0)